# Qdrant Vector Database Utilities

Generic, reusable Qdrant vector database functions that can be used by any pipeline.
Supports three modes: local (disk), memory (in-memory), and docker (containerized).

## Installation and Imports

In [9]:
# Install required packages
import subprocess
import sys

packages = [
    'qdrant-client',
    'numpy',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ Packages installed")

✓ Packages installed


In [10]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct, HnswConfigDiff

## Configuration Class

In [11]:
@dataclass
class QdrantConfig:
    """
    Configuration for Qdrant vector database
    
    Supports both single-vector and multi-vector modes.
    
    Example single-vector:
        config = QdrantConfig(
            embedding_dimension=384,
            distance="COSINE"
        )
    
    Example multi-vector:
        config = QdrantConfig(
            vectors_config={
                "text": {"dimension": 384, "distance": "COSINE"},
                "image": {"dimension": 512, "distance": "DOT"}
            }
        )
    """
    
    # Database mode: 'docker', 'memory', or 'local'
    db_mode: str = "local"
    
    # Local disk path (used when db_mode = "local")
    local_path: str = "./local_qdrant"
    
    # Docker connection settings
    docker_url: str = "http://localhost:6333"
    
    # Collection settings
    collection_name: str = "documents"
    
    # Single-vector mode (used if vectors_config is None)
    embedding_dimension: int = 384
    distance: str = "COSINE"  # COSINE, DOT, MANHATTAN, EUCLID
    
    # Multi-vector mode (overrides single-vector if specified)
    # Format: {"vector_name": {"dimension": int, "distance": str}}
    vectors_config: Optional[Dict[str, Dict]] = None
    
    # HNSW Index Parameters
    # Example: {"m": 16, "ef_construct": 200, "full_scan_threshold": 10000, "on_disk": False}
    hnsw_config: Optional[Dict] = None
    
    # Search-time parameters
    # Example: {"ef": 100}
    search_params: Optional[Dict] = None
    
    # Retrieval settings
    top_k: int = 5

## QdrantVectorDB Class

In [ ]:
class QdrantVectorDB:
    """
    Generic Qdrant Vector Database for storage and retrieval.
    
    This class handles:
    - Storing pre-computed embeddings (single or multi-vector)
    - Retrieving similar documents via vector search
    - Managing collections
    
    NOTE: This class does NOT handle embedding computation.
    Embeddings should be computed in a separate pipeline step and passed to this class.
    """
    
    def __init__(self, config: QdrantConfig):
        """
        Initialize the vector database with a config.
        
        Args:
            config: QdrantConfig object specifying db mode, collection settings, vector config, etc.
        """
        self.config = config
        self.client = self._initialize_client()
    
    def _initialize_client(self) -> QdrantClient:
        """
        Initialize Qdrant client based on the configured mode.
        Supports three modes: local (disk), memory (in-memory), docker (remote).
        """
        mode = self.config.db_mode.lower()
        
        if mode == "docker":
            print(f"Connecting to Qdrant Docker at {self.config.docker_url}...")
            client = QdrantClient(url=self.config.docker_url)
            print("✓ Connected to Qdrant Docker")
        elif mode == "memory":
            print("Initializing in-memory Qdrant...")
            client = QdrantClient(":memory:")
            print("✓ In-memory Qdrant initialized")
        elif mode == "local":
            print(f"Initializing local Qdrant at {self.config.local_path}...")
            client = QdrantClient(path=self.config.local_path)
            print(f"✓ Local Qdrant initialized")
        else:
            raise ValueError(f"Invalid mode: {mode}. Use 'docker', 'memory', or 'local'")
        
        return client
    
    def _get_distance_metric(self, distance_str: str) -> Distance:
        """Convert distance string to Qdrant Distance enum."""
        distance_map = {
            "COSINE": Distance.COSINE,
            "DOT": Distance.DOT,
            "MANHATTAN": Distance.MANHATTAN,
            "EUCLID": Distance.EUCLID,
        }
        return distance_map.get(distance_str.upper(), Distance.COSINE)
    
    def _build_vectors_config(self) -> Dict:
        """
        Build vectors configuration for collection creation.
        Handles both single-vector and multi-vector modes.
        """
        if self.config.vectors_config:
            # Multi-vector mode: create VectorParams for each named vector
            vectors_config = {}
            for vector_name, params in self.config.vectors_config.items():
                dimension = params.get("dimension", 384)
                distance_str = params.get("distance", "COSINE")
                distance = self._get_distance_metric(distance_str)
                vectors_config[vector_name] = VectorParams(
                    size=dimension,
                    distance=distance
                )
            return vectors_config
        else:
            # Single-vector mode
            distance = self._get_distance_metric(self.config.distance)
            return VectorParams(
                size=self.config.embedding_dimension,
                distance=distance
            )
    
    def create_collection(self, collection_name: str = None, force_recreate: bool = False) -> None:
        """
        Create a collection with configured vectors and HNSW index parameters.
        
        Args:
            collection_name: Collection name (uses config if None)
            force_recreate: If True, delete existing collection first
            
        Example:
            vdb.create_collection(force_recreate=True)
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        try:
            self.client.get_collection(collection_name)
            if force_recreate:
                print(f"Deleting existing collection '{collection_name}'...")
                self.client.delete_collection(collection_name)
            else:
                print(f"Collection '{collection_name}' already exists.")
                return
        except:
            pass
        
        vectors_config = self._build_vectors_config()
        
        # Build HNSW config if provided
        hnsw_config = None
        if self.config.hnsw_config:
            hnsw_config = HnswConfigDiff(**self.config.hnsw_config)
        
        self.client.create_collection(
            collection_name=collection_name,
            vectors_config=vectors_config,
            hnsw_config=hnsw_config
        )
        print(f"✓ Collection '{collection_name}' created")
        if self.config.vectors_config:
            print(f"  Mode: Multi-vector with {len(self.config.vectors_config)} vector fields")
        else:
            print(f"  Mode: Single-vector ({self.config.embedding_dimension}D, {self.config.distance})")
    
    def delete_collection(self, collection_name: str = None) -> None:
        """
        Delete a collection and all its documents.
        Use clear_collection() if you want to keep the collection structure.
        
        Args:
            collection_name: Collection name (uses config if None)
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        try:
            self.client.delete_collection(collection_name)
            print(f"✓ Collection '{collection_name}' deleted")
        except Exception as e:
            print(f"Error deleting collection: {e}")
    
    def index_documents(self,
                       documents: List[Dict],
                       collection_name: str = None,
                       batch_size: int = 100) -> int:
        """
        Index pre-computed embeddings into the collection.
        Documents are batched for efficiency.
        
        For SINGLE-VECTOR mode, each document should have:
            - 'id': unique identifier
            - 'embedding': pre-computed vector (np.ndarray or list)
            - 'text': optional, original text for reference
            - 'metadata': optional, additional fields
        
        For MULTI-VECTOR mode, each document should have:
            - 'id': unique identifier
            - 'embeddings': dict with vector names as keys
                Example: {'text': embedding1, 'image': embedding2}
            - 'text': optional, original text
            - 'metadata': optional, additional fields
        
        Args:
            documents: List of document dicts with embeddings
            collection_name: Collection name (uses config if None)
            batch_size: Documents per batch (default 100)
            
        Returns:
            int: Total number of documents indexed
            
        Example:
            docs = [
                {'id': 0, 'embedding': [...], 'text': 'Doc 1'},
                {'id': 1, 'embedding': [...], 'text': 'Doc 2'},
            ]
            count = vdb.index_documents(docs)
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        points = []
        total_indexed = 0
        
        for idx, doc in enumerate(documents):
            doc_id = doc.get('id', idx)
            
            # Determine if single or multi-vector mode
            if self.config.vectors_config:
                # Multi-vector mode
                if 'embeddings' not in doc or not isinstance(doc['embeddings'], dict):
                    print(f"Warning: Document {idx} missing 'embeddings' dict, skipping...")
                    continue
                
                # Convert embeddings dict
                vectors = {}
                for vector_name, embedding in doc['embeddings'].items():
                    if isinstance(embedding, np.ndarray):
                        vectors[vector_name] = embedding.tolist()
                    else:
                        vectors[vector_name] = embedding
            else:
                # Single-vector mode
                if 'embedding' not in doc:
                    print(f"Warning: Document {idx} missing 'embedding', skipping...")
                    continue
                
                embedding = doc['embedding']
                if isinstance(embedding, np.ndarray):
                    vectors = embedding.tolist()
                else:
                    vectors = embedding
            
            point = PointStruct(
                id=doc_id,
                vector=vectors,
                payload={
                    'text': doc.get('text', ''),
                    **doc.get('metadata', {})
                }
            )
            points.append(point)
            
            if (idx + 1) % batch_size == 0:
                try:
                    self.client.upsert(
                        collection_name=collection_name,
                        points=points
                    )
                    total_indexed += len(points)
                    print(f"Indexed {total_indexed} documents...")
                    points = []
                except Exception as e:
                    print(f"Error upserting batch: {e}")
        
        if points:
            try:
                self.client.upsert(
                    collection_name=collection_name,
                    points=points
                )
                total_indexed += len(points)
            except Exception as e:
                print(f"Error upserting final batch: {e}")
        
        print(f"✓ Total {total_indexed} documents indexed")
        return total_indexed
    
    def retrieve(self,
                query_embedding,
                collection_name: str = None,
                top_k: int = None,
                query_vector_name: str = None) -> List[Dict]:
        """
        Retrieve the top-k most similar documents using a pre-computed query embedding.
        
        Args:
            query_embedding: Pre-computed query embedding (np.ndarray or list)
            collection_name: Name of the collection. Uses config name if None.
            top_k: Number of results. Uses config value if None.
            query_vector_name: For multi-vector mode, which vector field to search
            
        Returns:
            List[Dict]: Retrieved documents with scores and metadata
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        if top_k is None:
            top_k = self.config.top_k
        
        # Convert numpy array to list if needed
        if isinstance(query_embedding, np.ndarray):
            query_embedding = query_embedding.tolist()
        
        # For multi-vector collections, specify which vector field to use
        vector_field = None
        if self.config.vectors_config:
            vector_field = query_vector_name or list(self.config.vectors_config.keys())[0]
        
        try:
            search_results = self.client.query_points(
                collection_name=collection_name,
                query=query_embedding,
                using=vector_field,
                limit=top_k,
                search_params=self.config.search_params
            ).points
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []
        
        retrieved_docs = []
        for result in search_results:
            retrieved_docs.append({
                'id': result.id,
                'score': result.score,
                'payload': result.payload,
                'text': result.payload.get('text', '')
            })
        
        return retrieved_docs
    
    def retrieve_by_ids(self,
                       doc_ids: List[int],
                       collection_name: str = None) -> List[Dict]:
        """
        Retrieve specific documents by their IDs.
        Useful when you already know which documents you want (like in a reranking pipeline).
        
        Args:
            doc_ids: List of document IDs to retrieve
            collection_name: Collection name (uses config if None)
            
        Returns:
            List[Dict]: Retrieved documents with payloads
            
        Example:
            docs = vdb.retrieve_by_ids([1, 3, 5])
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        try:
            points = self.client.retrieve(
                collection_name=collection_name,
                ids=doc_ids
            )
            
            docs = []
            for point in points:
                docs.append({
                    'id': point.id,
                    'payload': point.payload,
                    'text': point.payload.get('text', '')
                })
            return docs
        except Exception as e:
            print(f"Error retrieving by IDs: {e}")
            return []
    
    def count_documents(self, collection_name: str = None) -> int:
        """
        Get the total number of documents in a collection.
        
        Args:
            collection_name: Collection name (uses config if None)
            
        Returns:
            int: Total document count
            
        Example:
            total = vdb.count_documents()
            print(f"Collection has {total} documents")
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        try:
            info = self.client.get_collection(collection_name)
            return info.points_count if info else 0
        except:
            return 0
    
    def get_collection_info(self, collection_name: str = None) -> Dict:
        """
        Get detailed information about a collection.
        Returns metadata like points count, vector size, distance metrics, etc.
        
        Args:
            collection_name: Collection name (uses config if None)
            
        Returns:
            Dict: Collection metadata and statistics
            
        Example:
            info = vdb.get_collection_info()
            print(f"Vectors dimension: {info.config.params.vectors['text'].size}")
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        try:
            return self.client.get_collection(collection_name)
        except Exception as e:
            print(f"Error retrieving collection info: {e}")
            return {}
    
    def clear_collection(self, collection_name: str = None) -> None:
        """
        Delete all documents from a collection (but keep the collection itself).
        Useful for resetting/reindexing without recreating from scratch.
        
        Args:
            collection_name: Collection name (uses config if None)
            
        Example:
            vdb.clear_collection()  # Clear all documents, keep structure
        """
        if collection_name is None:
            collection_name = self.config.collection_name
        
        self.delete_collection(collection_name)
        self.create_collection(collection_name)
        print(f"✓ Collection '{collection_name}' cleared")

## Usage Examples

### Single-Vector Mode (Simple Case)
```python
# Create config for single-vector collection
config = QdrantConfig(
    db_mode="memory",
    collection_name="documents",
    embedding_dimension=384,
    distance="COSINE"
)

# Initialize database
vdb = QdrantVectorDB(config)
vdb.create_collection(force_recreate=True)

# Index documents with embeddings
docs = [
    {'id': 0, 'embedding': np.random.randn(384), 'text': 'Document 1'},
    {'id': 1, 'embedding': np.random.randn(384), 'text': 'Document 2'},
]
vdb.index_documents(docs)

# Retrieve similar documents
query_emb = np.random.randn(384)
results = vdb.retrieve(query_emb, top_k=5)
```

### Multi-Vector Mode (Advanced)
```python
# Config with text AND image vectors
config = QdrantConfig(
    db_mode="memory",
    collection_name="multimodal_docs",
    vectors_config={
        "text": {"dimension": 384, "distance": "COSINE"},
        "image": {"dimension": 512, "distance": "DOT"}
    }
)

vdb = QdrantVectorDB(config)
vdb.create_collection(force_recreate=True)

# Index documents with multiple embeddings
docs = [
    {
        'id': 0,
        'embeddings': {
            'text': np.random.randn(384),
            'image': np.random.randn(512)
        },
        'text': 'Document with image',
        'metadata': {'has_image': True}
    }
]
vdb.index_documents(docs)

# Search text field
results = vdb.retrieve(np.random.randn(384), query_vector_name="text", top_k=5)

# Search image field
results = vdb.retrieve(np.random.randn(512), query_vector_name="image", top_k=5)
```

### With Advanced Configuration
```python
# HNSW index tuning + search parameters
config = QdrantConfig(
    db_mode="memory",
    embedding_dimension=384,
    hnsw_config={
        "m": 16,               # connections per element
        "ef_construct": 200,   # dynamic list size during construction
        "full_scan_threshold": 10000
    },
    search_params={"ef": 100}  # search-time parameters
)
```
